In [17]:
import csv
import heapq
import time
import random
import os
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. CONFIGURACIÓN Y GENERACIÓN DEL ESCENARIO (CSV)
# ==========================================
CSV_FILE = 'escenario.csv'
SEED = 42
random.seed(SEED) # Semilla registrada para reproducibilidad

def generar_escenario_csv(filename):
    """
    Genera un archivo CSV de 12x12 representando la red logística.
    I = Inicio, F = Destino Final, E1,E2,E3 = Entregas, X = Obstáculo, 
    Números (1-5) = Costo de desplazamiento.
    """
    if os.path.exists(filename):
        print(f"El archivo {filename} ya existe. Cargando escenario...")
        return
    
    # Cuadrícula base con costos aleatorios entre 1 y 5
    grid = [[str(random.randint(1, 5)) for _ in range(12)] for _ in range(12)]
    
    # Agregar puntos clave
    grid[0][0] = 'I'      # Inicio
    grid[11][11] = 'F'    # Destino final
    grid[2][8] = 'E1'     # Entrega 1
    grid[8][2] = 'E2'     # Entrega 2
    grid[6][6] = 'E3'     # Entrega 3
    
    # Agregar obstáculos (X)
    obstaculos = [(1,1), (1,2), (3,5), (4,5), (5,5), (8,8), (9,8), (10,8)]
    for ox, oy in obstaculos:
        grid[ox][oy] = 'X'
        
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(grid)
    print(f"Escenario {filename} generado exitosamente.")

generar_escenario_csv(CSV_FILE)

# ==========================================
# 2. LECTURA DEL ENTORNO
# ==========================================
class EntornoLogistico:
    def __init__(self, csv_file):
        self.grid = []
        self.puntos_clave = {}
        
        with open(csv_file, mode='r') as file:
            reader = csv.reader(file)
            for i, row in enumerate(reader):
                self.grid.append(row)
                for j, val in enumerate(row):
                    if val in ['I', 'F', 'E1', 'E2', 'E3']:
                        self.puntos_clave[val] = (i, j)
                        
        self.filas = len(self.grid)
        self.cols = len(self.grid[0])
        
    def obtener_costo(self, x, y):
        val = self.grid[x][y]
        if val in ['I', 'F', 'E1', 'E2', 'E3']:
            return 1 # Costo base para puntos clave
        return int(val)
    
    def es_valido(self, x, y):
        return 0 <= x < self.filas and 0 <= y < self.cols and self.grid[x][y] != 'X'
    
    def obtener_vecinos(self, x, y):
        # Movimientos permitidos: Arriba, Abajo, Izquierda, Derecha
        movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        vecinos = []
        for mx, my in movimientos:
            nx, ny = x + mx, y + my
            if self.es_valido(nx, ny):
                vecinos.append((nx, ny))
        return vecinos

entorno = EntornoLogistico(CSV_FILE)

# ==========================================
# 3. MÓDULO A: BÚSQUEDA HEURÍSTICA (Greedy y A*)
# ==========================================
def heuristica_manhattan(a, b):
    """
    H(n): Distancia Manhattan.
    Es admisible porque nunca sobreestima el costo real en una cuadrícula
    donde solo te puedes mover en 4 direcciones y el costo mínimo es 1.
    """
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def busqueda_heuristica(entorno, inicio, objetivo, algoritmo="A*"):
    """
    Implementa A* (f(n) = g(n) + h(n)) o Greedy (f(n) = h(n)).
    Retorna la ruta, el costo total, nodos expandidos y tiempo de ejecución.
    """
    start_time = time.time()
    
    frontera = []
    heapq.heappush(frontera, (0, inicio))
    
    costo_acumulado = {inicio: 0}
    padres = {inicio: None}
    nodos_expandidos = 0
    
    while frontera:
        _, actual = heapq.heappop(frontera)
        nodos_expandidos += 1
        
        if actual == objetivo:
            break
            
        for vecino in entorno.obtener_vecinos(*actual):
            # Costo de moverse al vecino
            nuevo_costo = costo_acumulado[actual] + entorno.obtener_costo(*vecino)
            
            if vecino not in costo_acumulado or nuevo_costo < costo_acumulado[vecino]:
                costo_acumulado[vecino] = nuevo_costo
                padres[vecino] = actual
                
                h_n = heuristica_manhattan(vecino, objetivo)
                if algoritmo == "A*":
                    f_n = nuevo_costo + h_n  # f(n) = g(n) + h(n)
                else: # Greedy
                    f_n = h_n                # f(n) = h(n)
                    
                heapq.heappush(frontera, (f_n, vecino))
                
    # Reconstruir ruta
    ruta = []
    nodo = objetivo
    while nodo is not None:
        ruta.append(nodo)
        nodo = padres.get(nodo)
    ruta.reverse()
    
    tiempo_ejec = time.time() - start_time
    costo_final = costo_acumulado.get(objetivo, float('inf'))
    
    return {
        "algoritmo": algoritmo,
        "ruta": ruta,
        "costo": costo_final,
        "nodos_expandidos": nodos_expandidos,
        "tiempo": tiempo_ejec
    }

# Prueba del Módulo A: Ir de Inicio (I) a Entrega 1 (E1)
origen = entorno.puntos_clave['I']
destino = entorno.puntos_clave['E1']

resultado_astar = busqueda_heuristica(entorno, origen, destino, "A*")
resultado_greedy = busqueda_heuristica(entorno, origen, destino, "Greedy")

print(f"--- COMPARACIÓN DE BÚSQUEDA (De I a E1) ---")
print(f"A*     -> Costo: {resultado_astar['costo']}, Nodos Expandidos: {resultado_astar['nodos_expandidos']}, Tiempo: {resultado_astar['tiempo']:.5f}s")
print(f"Greedy -> Costo: {resultado_greedy['costo']}, Nodos Expandidos: {resultado_greedy['nodos_expandidos']}, Tiempo: {resultado_greedy['tiempo']:.5f}s")

El archivo escenario.csv ya existe. Cargando escenario...
--- COMPARACIÓN DE BÚSQUEDA (De I a E1) ---
A*     -> Costo: 19, Nodos Expandidos: 29, Tiempo: 0.00011s
Greedy -> Costo: 20, Nodos Expandidos: 11, Tiempo: 0.00003s


In [18]:
import numpy as np
import pandas as pd
import random
import csv
import heapq
import time


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CSV_FILENAME = 'mapa_logistico.csv'

def generar_csv_escenario():
    """Genera el mapa 12x12 con costos y obstáculos"""
    grid = [['1' for _ in range(12)] for _ in range(12)]
    # Zonas de alto costo
    for r in range(3, 6):
        for c in range(3, 6): grid[r][c] = '3'
    for r in range(8, 11):
        for c in range(8, 11): grid[r][c] = '5'
    
    # Obstáculos
    obstaculos = [(2, 2), (2, 3), (2, 4), (6, 8), (7, 8), (8, 8), (10, 1), (10, 2)]
    for r, c in obstaculos: grid[r][c] = 'X'
    
    # Puntos clave
    grid[0][0] = 'S'
    grid[11][11] = 'F'
    grid[1][10] = 'D1'
    grid[9][4] = 'D2'
    grid[5][11] = 'D3'
    
    with open(CSV_FILENAME, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(grid)

generar_csv_escenario()

class EntornoLogistico:
    def __init__(self, filepath):
        self.filepath = filepath
        self.grid = []
        self.inicio = None
        self.final = None
        self.entregas = {}
        self.obstaculos = set()
        self.cargar_mapa()
        
    def cargar_mapa(self):
        with open(self.filepath, mode='r') as file:
            reader = csv.reader(file)
            for r_idx, row in enumerate(reader):
                self.grid.append(row)
                for c_idx, val in enumerate(row):
                    pos = (r_idx, c_idx)
                    if val == 'S': self.inicio = pos
                    elif val == 'F': self.final = pos
                    elif val.startswith('D'): self.entregas[val] = pos
                    elif val == 'X': self.obstaculos.add(pos)

    def obtener_costo(self, pos):
        r, c = pos
        val = self.grid[r][c]
        if val in ['S', 'F', 'X'] or val.startswith('D'): return 1
        return int(val)

    def es_valida(self, pos):
        r, c = pos
        if 0 <= r < 12 and 0 <= c < 12 and pos not in self.obstaculos: return True
        return False

# Inicializamos el entorno para que esté disponible
entorno = EntornoLogistico(CSV_FILENAME)


# --- FASE 2: BÚSQUEDA HEURÍSTICA (GBFS vs A*) ---
def obtener_vecinos(entorno, pos):
    r, c = pos
    direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    vecinos = []
    for dr, dc in direcciones:
        n_pos = (r + dr, c + dc)
        if entorno.es_valida(n_pos):
            vecinos.append(n_pos)
    return vecinos

def heuristica_manhattan(actual, objetivo):
    """
    Justificación: Admisible porque en una cuadrícula con movimientos ortogonales, 
    la distancia de Manhattan nunca sobreestima el costo real mínimo (que es 1 por paso).
    """
    return abs(actual[0] - objetivo[0]) + abs(actual[1] - objetivo[1])

def greedy_bfs(entorno, inicio, objetivo):
    inicio_tiempo = time.perf_counter()
    frontera = []
    # Prioridad guiada solo por h(n)
    heapq.heappush(frontera, (heuristica_manhattan(inicio, objetivo), inicio, [inicio], 0))
    visitados = set()
    nodos_expandidos = 0
    
    while frontera:
        _, actual, ruta, costo_real = heapq.heappop(frontera)
        nodos_expandidos += 1
        if actual == objetivo:
            return ruta, costo_real, nodos_expandidos, time.perf_counter() - inicio_tiempo
        if actual in visitados: continue
        visitados.add(actual)
        
        for vecino in obtener_vecinos(entorno, actual):
            if vecino not in visitados:
                nuevo_costo = costo_real + entorno.obtener_costo(vecino)
                h_n = heuristica_manhattan(vecino, objetivo)
                heapq.heappush(frontera, (h_n, vecino, ruta + [vecino], nuevo_costo))
    return None, 0, nodos_expandidos, time.perf_counter() - inicio_tiempo

def a_star(entorno, inicio, objetivo):
    inicio_tiempo = time.perf_counter()
    frontera = []
    heapq.heappush(frontera, (heuristica_manhattan(inicio, objetivo), 0, inicio, [inicio]))
    costo_g_minimo = {inicio: 0}
    nodos_expandidos = 0
    
    while frontera:
        f_n, costo_g, actual, ruta = heapq.heappop(frontera)
        nodos_expandidos += 1
        if actual == objetivo:
            return ruta, costo_g, nodos_expandidos, time.perf_counter() - inicio_tiempo
            
        for vecino in obtener_vecinos(entorno, actual):
            nuevo_costo_g = costo_g + entorno.obtener_costo(vecino)
            if vecino not in costo_g_minimo or nuevo_costo_g < costo_g_minimo[vecino]:
                costo_g_minimo[vecino] = nuevo_costo_g
                # Prioridad guiada por f(n) = g(n) + h(n)
                f_n = nuevo_costo_g + heuristica_manhattan(vecino, objetivo)
                heapq.heappush(frontera, (f_n, nuevo_costo_g, vecino, ruta + [vecino]))
    return None, 0, nodos_expandidos, time.perf_counter() - inicio_tiempo

def probar_escenarios():
    print("--- RESULTADOS FASE 2: GREEDY BFS vs A* ---\n")
    escenarios = [
        ("Inicio (S) a Entrega 1 (D1)", entorno.inicio, entorno.entregas['D1']),
        ("Entrega 1 (D1) a Entrega 2 (D2)", entorno.entregas['D1'], entorno.entregas['D2']),
        ("Entrega 2 (D2) a Entrega 3 (D3)", entorno.entregas['D2'], entorno.entregas['D3'])
    ]
    
    for nombre, origen, destino in escenarios:
        print(f"[*] Escenario: {nombre} | Origen: {origen} -> Destino: {destino}")
        ruta_g, costo_g, exp_g, time_g = greedy_bfs(entorno, origen, destino)
        ruta_a, costo_a, exp_a, time_a = a_star(entorno, origen, destino)
        
        print(f"    -> Greedy BFS [h(n)]   : Costo={costo_g}, Nodos expand={exp_g}, Tiempo={time_g:.6f}s")
        print(f"    -> A* Search  [f(n)]   : Costo={costo_a}, Nodos expand={exp_a}, Tiempo={time_a:.6f}s")
        if costo_g > costo_a: 
            print("    [!] A* encontró una ruta más óptima que Greedy BFS.")
        print("-" * 75)

# Ejecutamos las pruebas
probar_escenarios()

--- RESULTADOS FASE 2: GREEDY BFS vs A* ---

[*] Escenario: Inicio (S) a Entrega 1 (D1) | Origen: (0, 0) -> Destino: (1, 10)
    -> Greedy BFS [h(n)]   : Costo=11, Nodos expand=12, Tiempo=0.000084s
    -> A* Search  [f(n)]   : Costo=11, Nodos expand=22, Tiempo=0.000109s
---------------------------------------------------------------------------
[*] Escenario: Entrega 1 (D1) a Entrega 2 (D2) | Origen: (1, 10) -> Destino: (9, 4)
    -> Greedy BFS [h(n)]   : Costo=22, Nodos expand=16, Tiempo=0.000524s
    -> A* Search  [f(n)]   : Costo=14, Nodos expand=48, Tiempo=0.000210s
    [!] A* encontró una ruta más óptima que Greedy BFS.
---------------------------------------------------------------------------
[*] Escenario: Entrega 2 (D2) a Entrega 3 (D3) | Origen: (9, 4) -> Destino: (5, 11)
    -> Greedy BFS [h(n)]   : Costo=15, Nodos expand=12, Tiempo=0.000054s
    -> A* Search  [f(n)]   : Costo=11, Nodos expand=22, Tiempo=0.000092s
    [!] A* encontró una ruta más óptima que Greedy BFS.
-----

In [19]:
import itertools

# 8. MATRIZ DE DISTANCIAS PRECOMPUTADA
# Para no ejecutar A* miles de veces durante la optimización, 
# precalculamos el costo óptimo entre todos los puntos clave.
print("[*] Precomputando matriz de costos óptimos con A*...")
puntos_clave = {'S': entorno.inicio, 'F': entorno.final}
puntos_clave.update(entorno.entregas)

matriz_costos = {}
for origen_nombre, origen_pos in puntos_clave.items():
    matriz_costos[origen_nombre] = {}
    for destino_nombre, destino_pos in puntos_clave.items():
        if origen_nombre == destino_nombre:
            matriz_costos[origen_nombre][destino_nombre] = 0
        else:
            # Usamos A* de la celda anterior para garantizar el costo real
            _, costo, _, _ = a_star(entorno, origen_pos, destino_pos)
            matriz_costos[origen_nombre][destino_nombre] = costo

# 9. EVALUACIÓN DE RUTAS CADA VEZ QUE SE MUEVEN LAS ENTREGAS
def calcular_costo_ruta(orden_entregas):
    """Calcula el costo del recorrido completo: S -> (Entregas) -> F"""
    costo_total = matriz_costos['S'][orden_entregas[0]]
    for i in range(len(orden_entregas) - 1):
        costo_total += matriz_costos[orden_entregas[i]][orden_entregas[i+1]]
    costo_total += matriz_costos[orden_entregas[-1]]['F']
    return costo_total

# 10. HILL CLIMBING (BÚSQUEDA LOCAL)
def hill_climbing(solucion_inicial):
    actual = solucion_inicial
    costo_actual = calcular_costo_ruta(actual)
    
    while True:
        vecinos = []
        # Vecindad explícita: intercambiar 2 entregas de lugar (swap)
        for i in range(len(actual)):
            for j in range(i + 1, len(actual)):
                vecino = actual[:]
                vecino[i], vecino[j] = vecino[j], vecino[i]
                vecinos.append(vecino)
        
        mejor_vecino = None
        mejor_costo_vecino = float('inf')
        
        for vecino in vecinos:
            costo_vecino = calcular_costo_ruta(vecino)
            if costo_vecino < mejor_costo_vecino:
                mejor_costo_vecino = costo_vecino
                mejor_vecino = vecino
                
        # Criterio de aceptación: si el vecino mejora el costo, nos movemos ahí
        if mejor_costo_vecino < costo_actual:
            actual = mejor_vecino
            costo_actual = mejor_costo_vecino
        else:
            # Si ningún vecino es mejor, hemos llegado a un óptimo local (o meseta)
            break
            
    return actual, costo_actual

# 11. PRUEBA CON 10 SOLUCIONES INICIALES
print("\n--- FASE 3: BÚSQUEDA LOCAL (HILL CLIMBING) ---\n")
entregas_list = list(entorno.entregas.keys())

# Al ser 3 entregas (D1, D2, D3), hay 6 permutaciones reales posibles.
# Tomamos 10 muestras aleatorias (con repetición) para cumplir la rúbrica y ver variabilidad.
todas_permutaciones = list(itertools.permutations(entregas_list))
soluciones_iniciales = [list(random.choice(todas_permutaciones)) for _ in range(10)]

resultados_hc = []
for idx, sol_inicial in enumerate(soluciones_iniciales):
    sol_final, costo_final = hill_climbing(sol_inicial)
    costo_inicial = calcular_costo_ruta(sol_inicial)
    resultados_hc.append(costo_final)
    
    print(f"Iteración {idx+1}:")
    print(f"  Ruta Inicial : {sol_inicial} (Costo: {costo_inicial})")
    print(f"  Óptimo Local : {sol_final} (Costo: {costo_final})")
    print("-" * 60)

# Análisis de óptimos
print(f"\n[ Análisis de Resultados Hill Climbing ]")
print(f"Mejor costo global encontrado : {min(resultados_hc)}")
print(f"Peor costo (óptimo local)     : {max(resultados_hc)}")
if min(resultados_hc) == max(resultados_hc):
    print("Nota: El algoritmo siempre convergió al mismo óptimo sin importar el inicio.")
else:
    print("Nota: Hubo variabilidad, demostrando que Hill Climbing es sensible a la solución inicial.")

[*] Precomputando matriz de costos óptimos con A*...

--- FASE 3: BÚSQUEDA LOCAL (HILL CLIMBING) ---

Iteración 1:
  Ruta Inicial : ['D2', 'D3', 'D1'] (Costo: 40)
  Óptimo Local : ['D1', 'D3', 'D2'] (Costo: 36)
------------------------------------------------------------
Iteración 2:
  Ruta Inicial : ['D1', 'D3', 'D2'] (Costo: 36)
  Óptimo Local : ['D1', 'D3', 'D2'] (Costo: 36)
------------------------------------------------------------
Iteración 3:
  Ruta Inicial : ['D1', 'D3', 'D2'] (Costo: 36)
  Óptimo Local : ['D1', 'D3', 'D2'] (Costo: 36)
------------------------------------------------------------
Iteración 4:
  Ruta Inicial : ['D2', 'D3', 'D1'] (Costo: 40)
  Óptimo Local : ['D1', 'D3', 'D2'] (Costo: 36)
------------------------------------------------------------
Iteración 5:
  Ruta Inicial : ['D3', 'D1', 'D2'] (Costo: 44)
  Óptimo Local : ['D1', 'D3', 'D2'] (Costo: 36)
------------------------------------------------------------
Iteración 6:
  Ruta Inicial : ['D1', 'D2', 'D3']

In [20]:
import math
import time
import numpy as np

# --- FASE 4: METAHEURÍSTICAS ---

# 12. RECOCIDO SIMULADO (Simulated Annealing)
def recocido_simulado(sol_inicial, t_inicial=100.0, alpha=0.9, max_iter=50):
    """
    Enfriamiento: Multiplicamos la temperatura por alpha (enfriamiento geométrico).
    Regla de aceptación: Si el costo empeora, se acepta con probabilidad e^(-Delta / T).
    """
    actual = sol_inicial
    costo_actual = calcular_costo_ruta(actual)
    mejor_sol = actual[:]
    mejor_costo = costo_actual
    
    T = t_inicial
    evaluaciones = 1
    
    while T > 0.1 and evaluaciones < max_iter:
        # Generar vecino aleatorio (Swap)
        vecino = actual[:]
        i, j = random.sample(range(len(vecino)), 2)
        vecino[i], vecino[j] = vecino[j], vecino[i]
        
        costo_vecino = calcular_costo_ruta(vecino)
        evaluaciones += 1
        
        delta = costo_vecino - costo_actual
        
        # Criterio de aceptación probabilística
        if delta < 0 or random.random() < math.exp(-delta / T):
            actual = vecino
            costo_actual = costo_vecino
            
            # Actualizar el mejor global
            if costo_actual < mejor_costo:
                mejor_costo = costo_actual
                mejor_sol = actual[:]
                
        # Enfriamiento
        T *= alpha
        
    return mejor_sol, mejor_costo, evaluaciones

# 13. ALGORITMO GENÉTICO
def inicializar_poblacion(tamano_pob, entregas):
    return [list(np.random.permutation(entregas)) for _ in range(tamano_pob)]

def seleccion_torneo(poblacion, costos, k=2):
    """Selecciona el mejor individuo de una muestra aleatoria de tamaño k."""
    seleccionados = random.sample(list(zip(poblacion, costos)), k)
    seleccionados.sort(key=lambda x: x[1])
    return seleccionados[0][0]

def cruza_orden(padre1, padre2):
    """Cruza para permutaciones (Order Crossover - OX1)."""
    hijo = [None] * len(padre1)
    # Al tener pocos genes (3 entregas), cruzamos un solo elemento directo
    punto = random.randint(0, len(padre1) - 1)
    hijo[punto] = padre1[punto]
    
    # Rellenar con los genes del padre 2 que falten
    p2_filtrado = [g for g in padre2 if g not in hijo]
    idx_p2 = 0
    for i in range(len(hijo)):
        if hijo[i] is None:
            hijo[i] = p2_filtrado[idx_p2]
            idx_p2 += 1
    return hijo

def mutacion_swap(individuo, prob_mutacion=0.1):
    """Intercambia dos entregas con una probabilidad dada."""
    if random.random() < prob_mutacion:
        i, j = random.sample(range(len(individuo)), 2)
        individuo[i], individuo[j] = individuo[j], individuo[i]
    return individuo

def algoritmo_genetico(entregas, tamano_pob=10, max_gen=20, prob_mut=0.2):
    """
    Tratamiento de Restricciones: Al usar cruza de orden (OX1) y mutación swap, 
    garantizamos que las soluciones generadas siempre sean permutaciones válidas 
    (no se duplican ni faltan entregas).
    Convergencia Prematura: Al ser un problema de 3 ciudades (6 rutas posibles), 
    la población pierde diversidad rápido. La tasa de mutación alta (0.2) 
    ayuda a explorar, pero es normal converger en la generación 1 o 2.
    """
    poblacion = inicializar_poblacion(tamano_pob, entregas)
    evaluaciones = 0
    mejor_global = None
    mejor_costo_global = float('inf')
    
    for generacion in range(max_gen):
        costos = [calcular_costo_ruta(ind) for ind in poblacion]
        evaluaciones += tamano_pob
        
        # Rastrear al mejor de la generación
        mejor_gen_costo = min(costos)
        if mejor_gen_costo < mejor_costo_global:
            mejor_costo_global = mejor_gen_costo
            mejor_global = poblacion[costos.index(mejor_gen_costo)][:]
            
        nueva_poblacion = []
        # Elitismo: guardamos al mejor
        nueva_poblacion.append(mejor_global)
        
        while len(nueva_poblacion) < tamano_pob:
            p1 = seleccion_torneo(poblacion, costos)
            p2 = seleccion_torneo(poblacion, costos)
            hijo = cruza_orden(p1, p2)
            hijo = mutacion_swap(hijo, prob_mut)
            nueva_poblacion.append(hijo)
            
        poblacion = nueva_poblacion
        
    return mejor_global, mejor_costo_global, evaluaciones


# 14. EXPERIMENTACIÓN Y COMPARACIÓN (20 Ejecuciones)
print("\n--- EJECUCIÓN DE 20 PRUEBAS POR MÉTODO ---")
entregas_base = list(entorno.entregas.keys())
resultados_sa = []
resultados_ag = []

for i in range(20):
    # Prueba Recocido Simulado
    sol_ini = list(np.random.permutation(entregas_base))
    t0 = time.perf_counter()
    _, costo_sa, eval_sa = recocido_simulado(sol_ini)
    t_sa = time.perf_counter() - t0
    resultados_sa.append({'costo': costo_sa, 'evals': eval_sa, 'tiempo': t_sa})
    
    # Prueba Algoritmo Genético
    t0 = time.perf_counter()
    _, costo_ag, eval_ag = algoritmo_genetico(entregas_base)
    t_ag = time.perf_counter() - t0
    resultados_ag.append({'costo': costo_ag, 'evals': eval_ag, 'tiempo': t_ag})

# 15. CÁLCULO DE MÉTRICAS ESTADÍSTICAS
def calcular_metricas(resultados):
    costos = [r['costo'] for r in resultados]
    evals = [r['evals'] for r in resultados]
    tiempos = [r['tiempo'] for r in resultados]
    
    return {
        'mejor_costo': min(costos),
        'promedio_costo': np.mean(costos),
        'std_costo': np.std(costos),
        'promedio_evals': np.mean(evals),
        'promedio_tiempo': np.mean(tiempos)
    }

metrics_sa = calcular_metricas(resultados_sa)
metrics_ag = calcular_metricas(resultados_ag)

print("\n[ Resultados: Recocido Simulado (20 ejecuciones) ]")
print(f"Mejor Costo : {metrics_sa['mejor_costo']}")
print(f"Promedio    : {metrics_sa['promedio_costo']:.2f}")
print(f"Desv. Estand: {metrics_sa['std_costo']:.4f}")
print(f"Evals Prom. : {metrics_sa['promedio_evals']:.1f}")
print(f"Tiempo Prom.: {metrics_sa['promedio_tiempo']:.6f} s")

print("\n[ Resultados: Algoritmo Genético (20 ejecuciones) ]")
print(f"Mejor Costo : {metrics_ag['mejor_costo']}")
print(f"Promedio    : {metrics_ag['promedio_costo']:.2f}")
print(f"Desv. Estand: {metrics_ag['std_costo']:.4f}")
print(f"Evals Prom. : {metrics_ag['promedio_evals']:.1f}")
print(f"Tiempo Prom.: {metrics_ag['promedio_tiempo']:.6f} s")


--- EJECUCIÓN DE 20 PRUEBAS POR MÉTODO ---

[ Resultados: Recocido Simulado (20 ejecuciones) ]
Mejor Costo : 36
Promedio    : 36.00
Desv. Estand: 0.0000
Evals Prom. : 50.0
Tiempo Prom.: 0.000104 s

[ Resultados: Algoritmo Genético (20 ejecuciones) ]
Mejor Costo : 36
Promedio    : 36.10
Desv. Estand: 0.4359
Evals Prom. : 200.0
Tiempo Prom.: 0.000995 s


In [21]:
import numpy as np
import pandas as pd

# --- FASE 5: TEORÍA DE JUEGOS ---

print("--- FASE 5: ANÁLISIS ESTRATÉGICO (JUEGO SIMULTÁNEO) ---\n")

# 16. DEFINICIÓN DE ESTRATEGIAS
estrategias_j1 = ["Ruta Segura", "Ruta Principal", "Ruta Secundaria"]
estrategias_j2 = ["Bloquear Principal", "Bloquear Secundaria", "No Bloquear"]

# 17. MATRIZ DE PAGOS (Utilidad Jugador 1, Utilidad Jugador 2)
# Valores interpretados como "Nivel de Beneficio" (mayor es mejor).
# J1 quiere hacer sus entregas eficientemente. J2 quiere entorpecer ahorrando sus recursos.
matriz_pagos = [
    [(4, 1), (4, 2), (5, 4)],  # J1: Ruta Segura
    [(2, 5), (6, 3), (3, 2)],  # J1: Ruta Principal
    [(3, 2), (5, 4), (4, 3)]   # J1: Ruta Secundaria
]

# Mostrar la matriz de manera legible
df_pagos = pd.DataFrame(matriz_pagos, index=estrategias_j1, columns=estrategias_j2)
print("[ Matriz de Pagos (J1, J2) ]")
print(df_pagos)
print("\n")

# 18. CÁLCULO DE MEJORES RESPUESTAS
mejores_respuestas_j1 = set() # (indice_fila, indice_columna)
mejores_respuestas_j2 = set()

# Para Jugador 1 (Busca el máximo pago en cada columna/estrategia de J2)
for j in range(3):
    pagos_columna = [matriz_pagos[i][j][0] for i in range(3)]
    max_pago = max(pagos_columna)
    for i in range(3):
        if matriz_pagos[i][j][0] == max_pago:
            mejores_respuestas_j1.add((i, j))

# Para Jugador 2 (Busca el máximo pago en cada fila/estrategia de J1)
for i in range(3):
    pagos_fila = [matriz_pagos[i][j][1] for j in range(3)]
    max_pago = max(pagos_fila)
    for j in range(3):
        if matriz_pagos[i][j][1] == max_pago:
            mejores_respuestas_j2.add((i, j))

print("[ Mejores Respuestas ]")
print("Jugador 1 (Agente):")
for i, j in mejores_respuestas_j1:
    print(f"  Si J2 juega '{estrategias_j2[j]}', J1 responde con '{estrategias_j1[i]}'")

print("\nJugador 2 (Competidor):")
for i, j in mejores_respuestas_j2:
    print(f"  Si J1 juega '{estrategias_j1[i]}', J2 responde con '{estrategias_j2[j]}'")

# 19. EQUILIBRIO DE NASH PURO
# Es la intersección donde ambos jugadores están jugando su mejor respuesta simultáneamente
equilibrios_nash = mejores_respuestas_j1.intersection(mejores_respuestas_j2)

print("\n[ Equilibrios de Nash Puros ]")
if equilibrios_nash:
    for i, j in equilibrios_nash:
        print(f"  -> ({estrategias_j1[i]}, {estrategias_j2[j]}) con pagos {matriz_pagos[i][j]}")
else:
    print("  No existen equilibrios de Nash puros en este juego.")

# 20. ANÁLISIS DE DOMINANCIA
def analizar_dominancia(pagos, estrategias, jugador_idx):
    """Revisa si alguna estrategia domina estrictamente a todas las demás."""
    # jugador_idx: 0 para J1 (filas), 1 para J2 (columnas)
    for e1 in range(3):
        domina_a_todas = True
        for e2 in range(3):
            if e1 == e2: continue
            # Revisar si e1 siempre es mayor que e2
            if jugador_idx == 0:
                if not all(matriz_pagos[e1][c][0] > matriz_pagos[e2][c][0] for c in range(3)):
                    domina_a_todas = False
            else:
                if not all(matriz_pagos[r][e1][1] > matriz_pagos[r][e2][1] for r in range(3)):
                    domina_a_todas = False
        if domina_a_todas:
            return estrategias[e1]
    return None

dom_j1 = analizar_dominancia(matriz_pagos, estrategias_j1, 0)
dom_j2 = analizar_dominancia(matriz_pagos, estrategias_j2, 1)

print("\n[ Análisis de Dominancia Estricta ]")
print(f"  Estrategia dominante J1: {dom_j1 if dom_j1 else 'Ninguna'}")
print(f"  Estrategia dominante J2: {dom_j2 if dom_j2 else 'Ninguna'}")

print("""
[ Interpretación Logística ]
El Equilibrio de Nash se da en (Ruta Segura, No Bloquear). Esto significa que 
si el Agente Logístico asume que el Competidor va a ahorrar recursos y no bloqueará, 
su mejor opción es tomar la Ruta Segura. A su vez, si el Competidor sabe que 
el Agente usará la Ruta Segura, lo que más le conviene es No Bloquear, ya que 
gastar recursos bloqueando las otras rutas no le dará ningún beneficio. 
Ambos se estancan en este escenario porque ninguno tiene incentivos para desviarse.
""")

--- FASE 5: ANÁLISIS ESTRATÉGICO (JUEGO SIMULTÁNEO) ---

[ Matriz de Pagos (J1, J2) ]
                Bloquear Principal Bloquear Secundaria No Bloquear
Ruta Segura                 (4, 1)              (4, 2)      (5, 4)
Ruta Principal              (2, 5)              (6, 3)      (3, 2)
Ruta Secundaria             (3, 2)              (5, 4)      (4, 3)


[ Mejores Respuestas ]
Jugador 1 (Agente):
  Si J2 juega 'No Bloquear', J1 responde con 'Ruta Segura'
  Si J2 juega 'Bloquear Secundaria', J1 responde con 'Ruta Principal'
  Si J2 juega 'Bloquear Principal', J1 responde con 'Ruta Segura'

Jugador 2 (Competidor):
  Si J1 juega 'Ruta Principal', J2 responde con 'Bloquear Principal'
  Si J1 juega 'Ruta Segura', J2 responde con 'No Bloquear'
  Si J1 juega 'Ruta Secundaria', J2 responde con 'Bloquear Secundaria'

[ Equilibrios de Nash Puros ]
  -> (Ruta Segura, No Bloquear) con pagos (5, 4)

[ Análisis de Dominancia Estricta ]
  Estrategia dominante J1: Ninguna
  Estrategia dominante J2: Nin

In [22]:
import math

print("--- FASE 6: BÚSQUEDA ADVERSARIA (MINIMAX Y PODA ALFA-BETA) ---\n")

# 21. CONFIGURACIÓN DEL JUEGO DE RECURSOS
# Valores de los recursos logísticos en disputa (ej. prioridad de paso, gasolina extra)
recursos_disponibles = [15, 10, 8, 5, 3, 2, 1] 
LIMITE_PROFUNDIDAD = 5

class NegociacionLogistica:
    def __init__(self):
        self.nodos_minimax = 0
        self.nodos_alfabeta = 0
        self.podas_realizadas = 0

    def es_terminal(self, recursos, profundidad):
        """Un estado es terminal si no hay recursos o si alcanzamos el límite de profundidad."""
        return len(recursos) == 0 or profundidad == 0

    def evaluar_estado(self, score_max, score_min):
        """Función de utilidad (heurística aproximada si se limita la profundidad)"""
        return score_max - score_min

    # 22. ALGORITMO MINIMAX TRADICIONAL
    def minimax(self, recursos, profundidad, is_max, score_max, score_min):
        self.nodos_minimax += 1
        
        if self.es_terminal(recursos, profundidad):
            return self.evaluar_estado(score_max, score_min)

        if is_max:
            mejor_valor = -math.inf
            for i, r in enumerate(recursos):
                # Generar el nuevo estado sin el recurso seleccionado
                nuevos_recursos = recursos[:i] + recursos[i+1:]
                valor = self.minimax(nuevos_recursos, profundidad - 1, False, score_max + r, score_min)
                mejor_valor = max(mejor_valor, valor)
            return mejor_valor
        else:
            mejor_valor = math.inf
            for i, r in enumerate(recursos):
                nuevos_recursos = recursos[:i] + recursos[i+1:]
                valor = self.minimax(nuevos_recursos, profundidad - 1, True, score_max, score_min + r)
                mejor_valor = min(mejor_valor, valor)
            return mejor_valor

    # 23. ALGORITMO MINIMAX CON PODA ALFA-BETA
    def alfabeta(self, recursos, profundidad, alpha, beta, is_max, score_max, score_min):
        self.nodos_alfabeta += 1
        
        if self.es_terminal(recursos, profundidad):
            return self.evaluar_estado(score_max, score_min)

        if is_max:
            mejor_valor = -math.inf
            for i, r in enumerate(recursos):
                nuevos_recursos = recursos[:i] + recursos[i+1:]
                valor = self.alfabeta(nuevos_recursos, profundidad - 1, alpha, beta, False, score_max + r, score_min)
                mejor_valor = max(mejor_valor, valor)
                alpha = max(alpha, mejor_valor)
                # Poda Beta
                if beta <= alpha:
                    self.podas_realizadas += 1
                    break
            return mejor_valor
        else:
            mejor_valor = math.inf
            for i, r in enumerate(recursos):
                nuevos_recursos = recursos[:i] + recursos[i+1:]
                valor = self.alfabeta(nuevos_recursos, profundidad - 1, alpha, beta, True, score_max, score_min + r)
                mejor_valor = min(mejor_valor, valor)
                beta = min(beta, mejor_valor)
                # Poda Alfa
                if beta <= alpha:
                    self.podas_realizadas += 1
                    break
            return mejor_valor

# 24. EJECUCIÓN Y COMPARACIÓN
juego = NegociacionLogistica()

print(f"[*] Recursos en la mesa para negociar: {recursos_disponibles}")
print(f"[*] Límite de profundidad establecido: {LIMITE_PROFUNDIDAD}\n")

# Ejecutar Minimax
valor_minimax = juego.minimax(recursos_disponibles, LIMITE_PROFUNDIDAD, True, 0, 0)

# Ejecutar Alfa-Beta
valor_alfabeta = juego.alfabeta(recursos_disponibles, LIMITE_PROFUNDIDAD, -math.inf, math.inf, True, 0, 0)

print("[ Resultados de la Búsqueda Adversaria ]")
print(f"  Valor óptimo Minimax   : {valor_minimax}")
print(f"  Valor óptimo Alfa-Beta : {valor_alfabeta}")

if valor_minimax == valor_alfabeta:
    print("  -> VERIFICADO: Ambos algoritmos calculan la misma utilidad óptima.")

print("\n[ Análisis de Rendimiento (Nodos Visitados) ]")
print(f"  Nodos explorados por Minimax   : {juego.nodos_minimax}")
print(f"  Nodos explorados por Alfa-Beta : {juego.nodos_alfabeta}")
print(f"  Total de podas realizadas      : {juego.podas_realizadas}")

ahorro = ((juego.nodos_minimax - juego.nodos_alfabeta) / juego.nodos_minimax) * 100
print(f"  -> La poda Alfa-Beta redujo el espacio de búsqueda en un {ahorro:.2f}%.")

--- FASE 6: BÚSQUEDA ADVERSARIA (MINIMAX Y PODA ALFA-BETA) ---

[*] Recursos en la mesa para negociar: [15, 10, 8, 5, 3, 2, 1]
[*] Límite de profundidad establecido: 5

[ Resultados de la Búsqueda Adversaria ]
  Valor óptimo Minimax   : 11
  Valor óptimo Alfa-Beta : 11
  -> VERIFICADO: Ambos algoritmos calculan la misma utilidad óptima.

[ Análisis de Rendimiento (Nodos Visitados) ]
  Nodos explorados por Minimax   : 3620
  Nodos explorados por Alfa-Beta : 246
  Total de podas realizadas      : 68
  -> La poda Alfa-Beta redujo el espacio de búsqueda en un 93.20%.
